In [64]:
# MEPS EDA (`data/MEPS`)

#Exploratory analysis of the MEPS public-use files in this folder:

#1. **Primary key** for the 2020 Prescribed Medicines file (`h220a.xlsx`)
#2. **Cross-file comparison** — file types, row grain, and column differences across years
#3. **Non-adherence analysis guide** — which files/columns to use (and which not to)

In [4]:
import pandas as pd
pd.set_option('display.max_columns', None)
df = pd.read_excel("../data/MEPS/h248a.xlsx")
df1 = df.head(5)
df1

,DUID,PID,DUPERSID,DRUGIDX,RXRECIDX,LINKIDX,PANEL,PURCHRD,RXBEGMM,RXBEGYRX,RXNAME,RXDRGNAM,RXNDC,RXQUANTY,RXFORM,RXFRMUNT,RXSTRENG,RXSTRUNT,RXDAYSUP,PHARTP1,PHARTP2,PHARTP3,PHARTP4,PHARTP5,PHARTP6,RXFLG,IMPFLAG,PCIMPFLG,DIABEQUIP,INPCFLG,TC1,TC1S1,TC1S1_1,TC1S1_2,TC1S2,TC1S2_1,TC1S3,TC1S3_1,TC2,TC2S1,TC2S1_1,TC2S1_2,TC2S2,TC3,TC3S1,TC3S1_1,RXSF23X,RXMR23X,RXMD23X,RXPV23X,RXVA23X,RXTR23X,RXOF23X,RXSL23X,RXWC23X,RXOT23X,RXXP23X,PERWT23F,VARSTR,VARPSU
0,2790002,101,2790002101,2790002101002,2790002101002303001,2790002101002303,27,3,-1,-1,BD PEN NEEDL,-15,8290320550,100.0,MISC,EA,-15,-15,-8,4,-1,-1,-1,-1,-1,1,4,2,1,1,-15,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,13.00,0.0,0.0,41.78,0.0,0.0,0.0,0.0,0.0,0.0,54.78,11664.426815,2019,1
1,2790002,101,2790002101,2790002101006,2790002101006403001,2790002101006403,27,4,5,2023,PREDNISONE,PREDNISONE,591544210,40.0,TABS,EA,10,MG,16,4,4,-1,-1,-1,-1,1,2,1,2,1,97,98,301,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,7.20,0.0,0.0,0.00,0.0,0.0,0.0,0.0,0.0,0.0,7.20,11664.426815,2019,1
2,2790002,101,2790002101,2790002101007,2790002101007403001,2790002101007403,27,4,7,2023,PREDNISONE,PREDNISONE,70954006020,10.0,TABS,EA,20,MG,5,4,4,-1,-1,-1,-1,1,5,1,2,1,97,98,301,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,2.39,0.0,0.0,10.93,0.0,0.0,0.0,0.0,0.0,0.0,13.32,11664.426815,2019,1
3,2790002,101,2790002101,2790002101008,2790002101008403001,2790002101008403,27,4,-15,2020,METFORMIN,METFORMIN,70010006305,60.0,TABS,EA,500,MG,30,4,4,-1,-1,-1,-1,1,2,1,2,1,358,99,214,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,6.22,0.0,0.0,0.00,0.0,0.0,0.0,0.0,0.0,0.0,6.22,11664.426815,2019,1
4,2790002,101,2790002101,2790002101008,2790002101008403002,2790002101008403,27,4,-15,2020,METFORMIN,METFORMIN,378718505,60.0,TABS,EA,500,MG,30,4,4,-1,-1,-1,-1,1,2,1,2,1,358,99,214,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,6.22,0.0,0.0,0.00,0.0,0.0,0.0,0.0,0.0,0.0,6.22,11664.426815,2019,1


In [65]:
import re
import subprocess
import sys
from collections import defaultdict
from pathlib import Path

import pandas as pd

try:
    import python_calamine  # noqa: F401
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "python-calamine", "-q"])

EXCEL_ENGINE = "calamine"  # required for H224.xlsx (openpyxl metadata error)


def resolve_meps_dir() -> Path:
    cwd = Path.cwd().resolve()
    candidates = [
        cwd / "data" / "MEPS",
        cwd / "MEPS",
        cwd,
        cwd.parent / "data" / "MEPS",
        cwd.parent.parent / "data" / "MEPS",
    ]
    for d in candidates:
        if d.exists() and any(d.glob("*.xlsx")):
            return d
    return candidates[0]


MEPS_DIR = resolve_meps_dir()
print(f"MEPS directory: {MEPS_DIR}")

FILE_TYPES = {
    "h220a.xlsx": ("2020", "Prescribed Medicines", "HC-220A"),
    "H224.xlsx": ("2020", "Full Year Consolidated", "HC-224"),
    "h222.xlsx": ("2020", "Medical Conditions", "HC-222"),
    "h223.xlsx": ("2020", "Person Round Plan", "HC-223"),
    "h229a.xlsx": ("2021", "Prescribed Medicines", "HC-229A"),
    "h231.xlsx": ("2021", "Medical Conditions", "HC-231"),
    "h232.xlsx": ("2021", "Person Round Plan", "HC-232"),
    "h233.xlsx": ("2021", "Full Year Consolidated", "HC-233"),
    "h239a.xlsx": ("2022", "Prescribed Medicines", "HC-239A"),
    "h241.xlsx": ("2022", "Medical Conditions", "HC-241"),
    "h242.xlsx": ("2022", "Person Round Plan", "HC-242"),
    "h243.xlsx": ("2022", "Full Year Consolidated", "HC-243"),
    "h248a.xlsx": ("2023", "Prescribed Medicines", "HC-248A"),
    "h249.xlsx": ("2023", "Medical Conditions", "HC-249"),
    "h250.xlsx": ("2023", "Person Round Plan", "HC-250"),
    "h251.xlsx": ("2023", "Full Year Consolidated", "HC-251"),
}


def list_meps_files() -> list[Path]:
    return sorted(p for p in MEPS_DIR.glob("*.xlsx") if not p.name.startswith("~$"))


def read_meps_columns(path: Path) -> tuple[list[str], str]:
    xl = pd.ExcelFile(path, engine=EXCEL_ENGINE)
    sheet = xl.sheet_names[0]
    cols = [str(c).strip() for c in pd.read_excel(path, sheet_name=sheet, nrows=0, engine=EXCEL_ENGINE).columns]
    return cols, sheet


def read_meps_row_count(path: Path, sheet: str) -> int:
    return len(pd.read_excel(path, sheet_name=sheet, usecols=[0], engine=EXCEL_ENGINE))

MEPS directory: /Users/friana/Medical Adherence/data/MEPS


## Part 1 — Primary key for `h220a.xlsx` (HC-220A, 2020 Prescribed Medicines)

Each row is one **acquisition-level** prescription record (a reported fill/purchase), not one person or one drug.

In [66]:
H220A_PATH = MEPS_DIR / "h220a.xlsx"
h220a = pd.read_excel(H220A_PATH, sheet_name="H220A", engine=EXCEL_ENGINE)
print(f"Loaded {H220A_PATH.name}: {h220a.shape[0]:,} rows × {h220a.shape[1]} columns")
h220a.head()

Loaded h220a.xlsx: 279,755 rows × 64 columns


,DUID,PID,DUPERSID,DRUGIDX,RXRECIDX,LINKIDX,PANEL,PURCHRD,RXBEGMM,RXBEGYRX,...,RXVA20X,RXTR20X,RXOF20X,RXSL20X,RXWC20X,RXOT20X,RXXP20X,PERWT20F,VARSTR,VARPSU
0,2320005,101,2320005101,2320005101001,2320005101001603001,2320005101001603,23,6,5,2019,...,0.0,0.0,0.0,0.0,0.0,0.0,8.95,8418.417067,2079,1
1,2320005,101,2320005101,2320005101001,2320005101001603002,2320005101001603,23,6,5,2019,...,0.0,0.0,0.0,0.0,0.0,0.0,8.95,8418.417067,2079,1
2,2320005,101,2320005101,2320005101001,2320005101001603003,2320005101001603,23,6,5,2019,...,0.0,0.0,0.0,0.0,0.0,0.0,8.95,8418.417067,2079,1
3,2320005,102,2320005102,2320005102001,2320005102001603001,2320005102001603,23,6,2,2017,...,0.0,0.0,0.0,0.0,0.0,0.0,196.62,5199.931866,2079,1
4,2320005,102,2320005102,2320005102001,2320005102001603002,2320005102001603,23,6,2,2017,...,0.0,0.0,0.0,0.0,0.0,0.0,75.37,5199.931866,2079,1


### Candidate identifiers

MEPS ships several related ID columns at the start of HC-220A. Only one should be treated as the row-level primary key.

In [67]:
id_cols = ["DUID", "PID", "DUPERSID", "DRUGIDX", "RXRECIDX", "LINKIDX"]

summary = (
    pd.DataFrame(
        {
            "column": id_cols,
            "non_null": [h220a[c].notna().sum() for c in id_cols],
            "n_unique": [h220a[c].nunique(dropna=False) for c in id_cols],
            "duplicate_rows": [h220a[c].duplicated().sum() for c in id_cols],
        }
    )
    .assign(
        pct_unique=lambda d: (100 * d["n_unique"] / len(h220a)).round(2),
        is_unique_key=lambda d: d["n_unique"] == len(h220a),
    )
)

summary

,column,non_null,n_unique,duplicate_rows,pct_unique,is_unique_key
0,DUID,279755,10156,269599,3.63,False
1,PID,279755,26,279729,0.01,False
2,DUPERSID,279755,15743,264012,5.63,False
3,DRUGIDX,279755,75622,204133,27.03,False
4,RXRECIDX,279755,279755,0,100.00,True
5,LINKIDX,279755,121443,158312,43.41,False


In [68]:
# Test composite keys that might look plausible but are not row-unique
composite_candidates = {
    "person (DUID + PID)": ["DUID", "PID"],
    "person-drug (DUPERSID + DRUGIDX)": ["DUPERSID", "DRUGIDX"],
    "linkage group (DUPERSID + LINKIDX)": ["DUPERSID", "LINKIDX"],
    "official record id (RXRECIDX)": ["RXRECIDX"],
}

composite_results = []
for label, cols in composite_candidates.items():
    n_rows = len(h220a)
    n_unique = h220a[cols].drop_duplicates().shape[0]
    composite_results.append(
        {
            "candidate_key": label,
            "columns": " + ".join(cols),
            "rows": n_rows,
            "unique_combinations": n_unique,
            "duplicate_rows": n_rows - n_unique,
            "is_primary_key": n_unique == n_rows,
        }
    )

pd.DataFrame(composite_results)

,candidate_key,columns,rows,unique_combinations,duplicate_rows,is_primary_key
0,person (DUID + PID),DUID + PID,279755,15743,264012,False
1,person-drug (DUPERSID + DRUGIDX),DUPERSID + DRUGIDX,279755,75622,204133,False
2,linkage group (DUPERSID + LINKIDX),DUPERSID + LINKIDX,279755,121443,158312,False
3,official record id (RXRECIDX),RXRECIDX,279755,279755,0,True


In [69]:
# Why RXRECIDX exists: same person + same drug + same round can still have multiple fills
example = (
    h220a.loc[
        h220a["DUPERSID"] == h220a.loc[0, "DUPERSID"],
        ["DUID", "PID", "DUPERSID", "DRUGIDX", "LINKIDX", "RXRECIDX", "PURCHRD", "RXNAME", "RXNDC"],
    ]
    .sort_values(["DRUGIDX", "RXRECIDX"])
    .head(6)
)

example

,DUID,PID,DUPERSID,DRUGIDX,LINKIDX,RXRECIDX,PURCHRD,RXNAME,RXNDC
0,2320005,101,2320005101,2320005101001,2320005101001603,2320005101001603001,6,METFORMIN,378718705
1,2320005,101,2320005101,2320005101001,2320005101001603,2320005101001603002,6,METFORMIN,378718705
2,2320005,101,2320005101,2320005101001,2320005101001603,2320005101001603003,6,METFORMIN,378718705


## Conclusion: primary key = `RXRECIDX`

**Choice:** `RXRECIDX` is the primary key for `h220a.xlsx`.

**Why this is the right call:**

1. **Official MEPS definition.** The HC-220A documentation states that `RXRECIDX` *"uniquely identifies each record on the file"* and labels it *"UNIQUE RX/PRESCRIBED MEDICINE IDENTIFIER"* ([MEPS HC-220A documentation](https://meps.ahrq.gov/data_stats/download_data/pufs/h220a/h220adoc.shtml)).

2. **Matches the file grain.** HC-220A is **acquisition-level**: one row per reported prescription fill/purchase. A single person can buy the same drug multiple times in the same survey round. Those rows share `DUPERSID`, `DRUGIDX`, and often `LINKIDX`, but each fill gets its own `RXRECIDX`.

3. **Empirical uniqueness.** In this file: 279,755 rows, 279,755 distinct `RXRECIDX` values, 0 nulls, 0 duplicates. No other single column — and no plausible composite without `RXRECIDX` — is row-unique.

4. **Built for that purpose.** MEPS constructs `RXRECIDX` as `LINKIDX` (16-character person-drug-round ID) plus a 3-digit enumeration suffix. That suffix is exactly what distinguishes multiple purchases that would otherwise look identical on linkage keys alone.

**Do not use as primary keys:**
- `DUPERSID` — identifies a **person**, not a prescription event.
- `DRUGIDX` — identifies a **person-drug** across rounds; one drug can have many fills.
- `LINKIDX` — identifies a **person-drug-round** for linking to conditions/other event files; still not one row per fill.

For joins to other MEPS files, use the appropriate foreign key (`DUPERSID`, `DRUGIDX`, or `LINKIDX`). For uniquely identifying rows **within** HC-220A, use `RXRECIDX`.

In [70]:
PRIMARY_KEY = "RXRECIDX"

assert h220a[PRIMARY_KEY].notna().all(), f"{PRIMARY_KEY} contains nulls"
assert h220a[PRIMARY_KEY].is_unique, f"{PRIMARY_KEY} is not unique"

print(f"Verified primary key: {PRIMARY_KEY}")
print(f"Rows: {len(h220a):,} | Unique {PRIMARY_KEY}: {h220a[PRIMARY_KEY].nunique():,}")

Verified primary key: RXRECIDX
Rows: 279,755 | Unique RXRECIDX: 279,755


## Part 2 — Cross-file comparison (all MEPS xlsx files)

There are **4 file types × 4 years (2020–2023)**. They are different MEPS tables — not the same dataset with different years only.

| File type | Row grain |
|---|---|
| Prescribed Medicines (`*a.xlsx`) | One prescription fill/purchase |
| Medical Conditions | One reported condition |
| Person Round Plan | One person-round insurance/plan record |
| Full Year Consolidated | One person (wide demographics + expenditures) |

In [71]:
# Load column names + row counts for every MEPS workbook in this folder
all_columns: dict[str, list[str]] = {}
file_catalog = []

for path in list_meps_files():
    cols, sheet = read_meps_columns(path)
    rows = read_meps_row_count(path, sheet)
    year, file_type, code = FILE_TYPES.get(path.name, ("?", "?", "?"))

    all_columns[path.name] = cols
    file_catalog.append(
        {
            "year": year,
            "code": code,
            "file_type": file_type,
            "file": path.name,
            "sheet": sheet,
            "rows": rows,
            "n_cols": len(cols),
        }
    )

catalog_df = (
    pd.DataFrame(file_catalog)
    .sort_values(["year", "file_type"])
    .reset_index(drop=True)
)

catalog_df

,year,code,file_type,file,sheet,rows,n_cols
0,2020,HC-224,Full Year Consolidated,H224.xlsx,H224,27805,1451
1,2020,HC-222,Medical Conditions,h222.xlsx,H222,80802,30
2,2020,HC-223,Person Round Plan,h223.xlsx,H223,45214,57
3,2020,HC-220A,Prescribed Medicines,h220a.xlsx,H220A,279755,64
4,2021,HC-233,Full Year Consolidated,h233.xlsx,H233,28336,1488
5,2021,HC-231,Medical Conditions,h231.xlsx,H231,94641,32
6,2021,HC-232,Person Round Plan,h232.xlsx,H232,50837,55
7,2021,HC-229A,Prescribed Medicines,h229a.xlsx,H229A,303394,66
8,2022,HC-243,Full Year Consolidated,h243.xlsx,H243,22431,1420
9,2022,HC-241,Medical Conditions,h241.xlsx,H241,83173,33


In [72]:
# Within each file type: how many columns are shared across all years?
TYPE_ORDER = [
    "Prescribed Medicines",
    "Medical Conditions",
    "Person Round Plan",
    "Full Year Consolidated",
]

within_type_rows = []
for file_type in TYPE_ORDER:
    files = sorted(
        [name for name, meta in FILE_TYPES.items() if meta[1] == file_type and name in all_columns],
        key=lambda n: FILE_TYPES[n][0],
    )
    if not files:
        continue

    col_sets = {name: set(all_columns[name]) for name in files}
    common = set.intersection(*col_sets.values())
    union = set.union(*col_sets.values())
    identical = len({tuple(all_columns[name]) for name in files}) == 1

    within_type_rows.append(
        {
            "file_type": file_type,
            "files": ", ".join(files),
            "n_files": len(files),
            "cols_min": min(len(all_columns[n]) for n in files),
            "cols_max": max(len(all_columns[n]) for n in files),
            "shared_all_years": len(common),
            "union_all_years": len(union),
            "identical_column_lists": identical,
        }
    )

within_type_df = pd.DataFrame(within_type_rows)
within_type_df

,file_type,files,n_files,cols_min,cols_max,shared_all_years,union_all_years,identical_column_lists
0,Prescribed Medicines,"h220a.xlsx, h229a.xlsx, h239a.xlsx, h248a.xlsx",4,60,66,48,102,False
1,Medical Conditions,"h222.xlsx, h231.xlsx, h241.xlsx, h249.xlsx",4,29,33,21,42,False
2,Person Round Plan,"h223.xlsx, h232.xlsx, h242.xlsx, h250.xlsx",4,55,57,48,66,False
3,Full Year Consolidated,"H224.xlsx, h233.xlsx, h243.xlsx, h251.xlsx",4,1374,1488,538,3864,False


In [73]:
# Columns that appear in only one year within a file type
year_only_rows = []

for file_type in TYPE_ORDER:
    files = sorted(
        [name for name, meta in FILE_TYPES.items() if meta[1] == file_type and name in all_columns],
        key=lambda n: FILE_TYPES[n][0],
    )
    if len(files) < 2:
        continue

    col_sets = {name: set(all_columns[name]) for name in files}
    common = set.intersection(*col_sets.values())

    for name in files:
        year = FILE_TYPES[name][0]
        only_here = sorted(col_sets[name] - common)
        if only_here:
            year_only_rows.append(
                {
                    "file_type": file_type,
                    "year": year,
                    "file": name,
                    "year_only_cols": len(only_here),
                    "examples": ", ".join(only_here[:8]) + (" ..." if len(only_here) > 8 else ""),
                }
            )

year_only_df = pd.DataFrame(year_only_rows)
year_only_df

,file_type,year,file,year_only_cols,examples
0,Prescribed Medicines,2020,h220a.xlsx,16,"PERWT20F, PHARTP7, PHARTP8, PHARTP9, PREGCAT, ..."
1,Prescribed Medicines,2021,h229a.xlsx,18,"PERWT21F, PHARTP10, PHARTP11, PHARTP7, PHARTP8..."
2,Prescribed Medicines,2022,h239a.xlsx,17,"PERWT22F, PHARTP10, PHARTP11, PHARTP7, PHARTP8..."
3,Prescribed Medicines,2023,h248a.xlsx,12,"PERWT23F, RXMD23X, RXMR23X, RXOF23X, RXOT23X, ..."
4,Medical Conditions,2020,h222.xlsx,9,"CRND6, CRND7, ERNUM, HHNUM, IPNUM, OBNUM, OPNU..."
5,Medical Conditions,2021,h231.xlsx,11,"CRND6, CRND7, CRND8, CRND9, ERCOND, HHCOND, IP..."
6,Medical Conditions,2022,h241.xlsx,12,"CCSR4X, CRND6, CRND7, CRND8, CRND9, ERCOND, HH..."
7,Medical Conditions,2023,h249.xlsx,8,"CCSR4X, ERCOND, HHCOND, IPCOND, OBCOND, OPCOND..."
8,Person Round Plan,2020,h223.xlsx,9,"ANNDEDCT, EVALCOV5, HOSPINSX, InsurPrivIDEX, M..."
9,Person Round Plan,2021,h232.xlsx,7,"ANNDEDCT, HOSPINSX, InsurPrivIDEX, MSUPINSX, P..."


In [74]:
# Cross-type overlap: 2020 files share very few column names
y2020_files = sorted(
    [name for name, meta in FILE_TYPES.items() if meta[0] == "2020" and name in all_columns],
    key=lambda n: FILE_TYPES[n][1],
)

cross_type_rows = []
for i, file_a in enumerate(y2020_files):
    for file_b in y2020_files[i + 1 :]:
        shared = sorted(set(all_columns[file_a]) & set(all_columns[file_b]))
        cross_type_rows.append(
            {
                "file_a": f"{FILE_TYPES[file_a][2]} ({file_a})",
                "file_b": f"{FILE_TYPES[file_b][2]} ({file_b})",
                "n_shared_cols": len(shared),
                "shared_cols": ", ".join(shared),
            }
        )

cross_type_df = pd.DataFrame(cross_type_rows)
cross_type_df

,file_a,file_b,n_shared_cols,shared_cols
0,HC-224 (H224.xlsx),HC-222 (h222.xlsx),7,"DUID, DUPERSID, PANEL, PERWT20F, PID, VARPSU, ..."
1,HC-224 (H224.xlsx),HC-223 (h223.xlsx),2,"DUPERSID, PANEL"
2,HC-224 (H224.xlsx),HC-220A (h220a.xlsx),7,"DUID, DUPERSID, PANEL, PERWT20F, PID, VARPSU, ..."
3,HC-222 (h222.xlsx),HC-223 (h223.xlsx),2,"DUPERSID, PANEL"
4,HC-222 (h222.xlsx),HC-220A (h220a.xlsx),7,"DUID, DUPERSID, PANEL, PERWT20F, PID, VARPSU, ..."
5,HC-223 (h223.xlsx),HC-220A (h220a.xlsx),2,"DUPERSID, PANEL"


In [75]:
# Sample leading columns by file type (2020) — shows each table has its own schema
sample_cols_rows = []

for name in y2020_files:
    cols = all_columns[name]
    sample_cols_rows.append(
        {
            "code": FILE_TYPES[name][2],
            "file": name,
            "rows": next(r["rows"] for r in file_catalog if r["file"] == name),
            "n_cols": len(cols),
            "first_8_cols": ", ".join(cols[:8]),
            "last_4_cols": ", ".join(cols[-4:]),
        }
    )

sample_cols_df = pd.DataFrame(sample_cols_rows)
sample_cols_df

,code,file,rows,n_cols,first_8_cols,last_4_cols
0,HC-224,H224.xlsx,27805,1451,"DUID, PID, DUPERSID, PANEL, FAMID31, FAMID42, ...","SAQWT20F, DIABW20F, VARSTR, VARPSU"
1,HC-222,h222.xlsx,80802,30,"DUID, PID, DUPERSID, CONDN, CONDIDX, PANEL, CO...","RXNUM, PERWT20F, VARSTR, VARPSU"
2,HC-223,h223.xlsx,45214,57,"EPCPIDX, DUPERSID, PHLDRIDX, ESTBIDX, EPRSIDX,...","ANNDEDCT, HSAACCT, UPRHMO, NAMECHNG"
3,HC-220A,h220a.xlsx,279755,64,"DUID, PID, DUPERSID, DRUGIDX, RXRECIDX, LINKID...","RXXP20X, PERWT20F, VARSTR, VARPSU"


### Cross-file takeaways

- **Different file types = different schemas.** Do not stack `h220a`, `h222`, `h223`, and `H224` like one table.
- **Same file type across years** keeps the same general structure, but column names change (year suffixes like `RXSF20X` → `RXSF23X`, plus some variables added/dropped).
- **Join keys between files:** `DUPERSID` (person), `LINKIDX` (Rx ↔ conditions), `RXRECIDX` (unique Rx row), `CONDIDX` (unique condition row).
- **For adherence work**, the prescription files are `h220a`, `h229a`, `h239a`, and `h248a`.

## Part 3 — Non-adherence analysis: which files and columns?

**Short answer:** use **3 file types per survey year**, not all 16 workbooks.

| Priority | File type | Example (2020) | Why |
|---|---|---|---|
| **Required** | Prescribed Medicines (`*a.xlsx`) | `h220a.xlsx` | One row = one fill/refill. This is where refill frequency and days-supply live. |
| **Strongly recommended** | Full Year Consolidated | `H224.xlsx` | Person-level covariates: demographics, insurance, cost barriers (`PMEDUP*`), health status proxies. |
| **Optional** | Medical Conditions | `h222.xlsx` | Comorbidity burden; link drugs to treated conditions via `LINKIDX`. |
| **Usually skip** | Person Round Plan | `h223.xlsx` | Insurance detail mostly duplicated in Consolidated; only needed for round-level plan changes. |

**Important limitation for your side-effects question:** MEPS has **no drug-specific side-effect variable** tied to a prescription row. You can proxy general symptoms at the **person** level (e.g. `ADMOOD42`, `ADPAIN42` in Consolidated), but that cannot cleanly prove "this drug caused side effects → fewer refills" without stronger assumptions or external data (e.g. CMS claims, EHR, or patient-reported adverse events).

In [76]:
# Matched file sets by survey year (use these together — never mix years in one row)
YEAR_FILE_SETS = pd.DataFrame(
    [
        {"year": "2020", "prescribed_meds": "h220a.xlsx", "consolidated": "H224.xlsx", "conditions": "h222.xlsx", "person_plan": "h223.xlsx"},
        {"year": "2021", "prescribed_meds": "h229a.xlsx", "consolidated": "h233.xlsx", "conditions": "h231.xlsx", "person_plan": "h232.xlsx"},
        {"year": "2022", "prescribed_meds": "h239a.xlsx", "consolidated": "h243.xlsx", "conditions": "h241.xlsx", "person_plan": "h242.xlsx"},
        {"year": "2023", "prescribed_meds": "h248a.xlsx", "consolidated": "h251.xlsx", "conditions": "h249.xlsx", "person_plan": "h250.xlsx"},
    ]
)

analysis_tiers = pd.DataFrame(
    [
        {"tier": "Core (start here)", "files": "Prescribed Meds + Consolidated", "n_workbooks_per_year": 2, "use_when": "Refill counts, days supply, cost/insurance confounders"},
        {"tier": "Extended", "files": "+ Medical Conditions", "n_workbooks_per_year": 3, "use_when": "Condition burden, drug-indication context via LINKIDX"},
        {"tier": "Full MEPS folder", "files": "All 4 types × 4 years", "n_workbooks_per_year": 16, "use_when": "Not recommended for first pass — redundant columns, heavy joins"},
    ]
)

YEAR_FILE_SETS

,year,prescribed_meds,consolidated,conditions,person_plan
0,2020,h220a.xlsx,H224.xlsx,h222.xlsx,h223.xlsx
1,2021,h229a.xlsx,h233.xlsx,h231.xlsx,h232.xlsx
2,2022,h239a.xlsx,h243.xlsx,h241.xlsx,h242.xlsx
3,2023,h248a.xlsx,h251.xlsx,h249.xlsx,h250.xlsx


In [77]:
# Column roles for non-adherence analysis (2020 example; payment cols use year suffix 20, 21, ...)
ANALYSIS_YEAR = "2020"
rx_file = YEAR_FILE_SETS.loc[YEAR_FILE_SETS["year"] == ANALYSIS_YEAR, "prescribed_meds"].iloc[0]
cons_file = YEAR_FILE_SETS.loc[YEAR_FILE_SETS["year"] == ANALYSIS_YEAR, "consolidated"].iloc[0]
cond_file = YEAR_FILE_SETS.loc[YEAR_FILE_SETS["year"] == ANALYSIS_YEAR, "conditions"].iloc[0]

yr_suffix = ANALYSIS_YEAR[-2:]  # '20' for 2020

column_roles = pd.DataFrame(
    [
        # --- Prescribed Medicines: outcomes & drug identifiers ---
        {"file": rx_file, "role": "Primary key", "column": "RXRECIDX", "why": "Unique fill/refill row"},
        {"file": rx_file, "role": "Person join key", "column": "DUPERSID", "why": "Merge to Consolidated / Conditions"},
        {"file": rx_file, "role": "Drug grouping", "column": "DRUGIDX", "why": "Same person + same drug across fills"},
        {"file": rx_file, "role": "Condition link key", "column": "LINKIDX", "why": "Link Rx event to Medical Conditions file"},
        {"file": rx_file, "role": "Refill timing (round)", "column": "PURCHRD", "why": "Survey round the fill was obtained"},
        {"file": rx_file, "role": "Adherence — supply", "column": "RXDAYSUP", "why": "Days supplied per fill (imputed; check MEPS missing codes)"},
        {"file": rx_file, "role": "Adherence — quantity", "column": "RXQUANTY", "why": "Pills/units dispensed per fill"},
        {"file": rx_file, "role": "Drug identity", "column": "RXNAME / RXNDC / RXDRGNAM", "why": "Stratify by drug or class"},
        {"file": rx_file, "role": "Drug class", "column": "TC1, TC2, TC3 (+ sub-class cols)", "why": "Therapeutic category confounding"},
        {"file": rx_file, "role": "Affordability (fill-level)", "column": f"RXXP{yr_suffix}X", "why": "Out-of-pocket paid for this fill"},
        {"file": rx_file, "role": "Survey weight", "column": f"PERWT{yr_suffix}F", "why": "National estimates"},
        {"file": rx_file, "role": "Variance estimation", "column": "VARSTR, VARPSU", "why": "Complex survey design"},
        {"file": rx_file, "role": "Data quality", "column": "IMPFLAG, RXFLG", "why": "Imputed vs reported fills — filter/sensitivity analyses"},
        # --- Consolidated: person-level predictors / confounders ---
        {"file": cons_file, "role": "Person join key", "column": "DUPERSID", "why": "Merge person traits onto drug fills"},
        {"file": cons_file, "role": "Cost barrier (NOT side effects)", "column": "PMEDUP42, PMEDUP53", "why": "Problems paying for prescribed medicine"},
        {"file": cons_file, "role": "Symptom proxy (weak)", "column": "ADMOOD42, ADPAIN42, ADSLEEP42", "why": "Person-level SAQ symptoms — not drug-specific adverse events"},
        {"file": cons_file, "role": "Insurance", "column": f"INSCOV{yr_suffix}, MCARE{yr_suffix}, MCAID{yr_suffix}, PRIV{yr_suffix}", "why": "Coverage affects access/adherence"},
        {"file": cons_file, "role": "Demographics", "column": f"AGE{yr_suffix}X, SEX, RACETHX, POVCAT{yr_suffix}", "why": "Standard confounders"},
        {"file": cons_file, "role": "Person Rx spending", "column": f"RXTOT{yr_suffix}, RXEXP{yr_suffix}", "why": "Overall medication use intensity"},
        # --- Medical Conditions: optional ---
        {"file": cond_file, "role": "Condition key", "column": "CONDIDX", "why": "Unique condition row"},
        {"file": cond_file, "role": "Diagnosis", "column": "ICD10CDX, CCSR1X", "why": "Comorbidity / indication context"},
        {"file": cond_file, "role": "Rx link (2020)", "column": "RXNUM", "why": "2020: count of Rx events linked to condition; 2021+ use RXCOND flag instead"},
    ]
)

column_roles

,file,role,column,why
0,h220a.xlsx,Primary key,RXRECIDX,Unique fill/refill row
1,h220a.xlsx,Person join key,DUPERSID,Merge to Consolidated / Conditions
2,h220a.xlsx,Drug grouping,DRUGIDX,Same person + same drug across fills
3,h220a.xlsx,Condition link key,LINKIDX,Link Rx event to Medical Conditions file
4,h220a.xlsx,Refill timing (round),PURCHRD,Survey round the fill was obtained
5,h220a.xlsx,Adherence — supply,RXDAYSUP,Days supplied per fill (imputed; check MEPS mi...
6,h220a.xlsx,Adherence — quantity,RXQUANTY,Pills/units dispensed per fill
7,h220a.xlsx,Drug identity,RXNAME / RXNDC / RXDRGNAM,Stratify by drug or class
8,h220a.xlsx,Drug class,"TC1, TC2, TC3 (+ sub-class cols)",Therapeutic category confounding
9,h220a.xlsx,Affordability (fill-level),RXXP20X,Out-of-pocket paid for this fill


In [78]:
# What MEPS does NOT provide for a clean side-effects → refill analysis
cons_cols = all_columns.get(cons_file) or read_meps_columns(MEPS_DIR / cons_file)[0]
rx_cols = all_columns.get(rx_file) or read_meps_columns(MEPS_DIR / rx_file)[0]

side_effect_search = ["SIDE", "ADVERSE", "AE", "ADR", "EFFECT", "TOLER", "STOPDRUG", "RXTSTOP"]
direct_side_effect_hits = [c for c in rx_cols + cons_cols if any(term in c.upper() for term in side_effect_search)]

symptom_proxies = [c for c in cons_cols if c.startswith("AD") and c.endswith("42")][:20]
cost_barrier_cols = [c for c in cons_cols if "PMED" in c.upper()]

limitation_df = pd.DataFrame(
    [
        {"check": "Drug-specific side-effect column in Prescribed Meds", "found": any(c in rx_cols for c in direct_side_effect_hits), "notes": "No — Rx file is utilization/payment only"},
        {"check": "Exact pharmacy fill date per acquisition", "found": "RXBEGMM" in rx_cols, "notes": "Only start-month/year on therapy — not per-fill dates in the PUF"},
        {"check": "Direct adherence question (e.g. 'skipped doses')", "found": False, "notes": "Must derive from fill counts + RXDAYSUP (imperfect PDC)"},
        {"check": "Person-level symptom proxies in Consolidated", "found": len(symptom_proxies) > 0, "notes": f"Examples: {', '.join(symptom_proxies[:6])}"},
        {"check": "Cost-related non-adherence in Consolidated", "found": len(cost_barrier_cols) > 0, "notes": f"Examples: {', '.join(cost_barrier_cols)}"},
    ]
)

print("Direct side-effect column name matches:", direct_side_effect_hits or "(none)")
limitation_df

Direct side-effect column name matches: ['ADREST42', 'ADRNK542', 'ADRNK442', 'ADRATETRT42', 'ADRELTRT42', 'VAEV20', 'GVAEV20', 'HHAEXP20']


,check,found,notes
0,Drug-specific side-effect column in Prescribed...,False,No — Rx file is utilization/payment only
1,Exact pharmacy fill date per acquisition,True,Only start-month/year on therapy — not per-fil...
2,Direct adherence question (e.g. 'skipped doses'),False,Must derive from fill counts + RXDAYSUP (imper...
3,Person-level symptom proxies in Consolidated,True,"Examples: ADPROX42, ADSEX42, ADAGE42, ADGENH42..."
4,Cost-related non-adherence in Consolidated,True,"Examples: CHPMED42, PMEDIN31, PMEDIN42, PMEDIN..."


In [79]:
# Build adherence/refill outcomes from Prescribed Medicines (2020 example)
adherence_cols = [
    "DUPERSID", "DRUGIDX", "RXRECIDX", "LINKIDX", "PURCHRD",
    "RXNAME", "RXNDC", "RXDAYSUP", "RXQUANTY", "RXBEGMM", "RXBEGYRX",
    f"RXXP{yr_suffix}X", f"PERWT{yr_suffix}F", "IMPFLAG",
]

rx_events = pd.read_excel(
    MEPS_DIR / rx_file,
    sheet_name=0,
    usecols=lambda c: c in adherence_cols,
    engine=EXCEL_ENGINE,
)

# MEPS often codes missing / inapplicable as negative values — exclude from supply stats
valid_supply = rx_events["RXDAYSUP"].where(rx_events["RXDAYSUP"] > 0)

person_drug = (
    rx_events.groupby(["DUPERSID", "DRUGIDX", "RXNAME"], as_index=False)
    .agg(
        n_fills=("RXRECIDX", "nunique"),
        total_days_supplied=("RXDAYSUP", lambda s: s.where(s > 0).sum()),
        median_days_supplied=("RXDAYSUP", lambda s: s.where(s > 0).median()),
        n_rounds_with_fill=("PURCHRD", "nunique"),
        total_oop=(f"RXXP{yr_suffix}X", "sum"),
    )
)

person_drug["multiple_fills"] = person_drug["n_fills"] > 1

print("Person-drug fill count distribution:")
display(person_drug["n_fills"].describe().to_frame("n_fills"))

print(f"Share of person-drug pairs with >1 fill: {person_drug['multiple_fills'].mean():.1%}")
person_drug.sort_values("n_fills", ascending=False).head(10)

Person-drug fill count distribution:


,n_fills
count,78609.000000
mean,3.558816
std,3.121049
min,1.000000
25%,1.000000
50%,3.000000
75%,5.000000
max,106.000000


Share of person-drug pairs with >1 fill: 66.6%


,DUPERSID,DRUGIDX,RXNAME,n_fills,total_days_supplied,median_days_supplied,n_rounds_with_fill,total_oop,multiple_fills
45341,2466612102,2466612102009,ONETOUCH,106,5300.0,50.0,3,881.92,True
73276,2577708104,2577708104001,NARCOTIC ANALGESICS,60,0.0,NaN,3,982.75,True
38707,2464502101,2464502101005,NARCOTIC ANALGESICS,40,0.0,NaN,3,673.59,True
10315,2324166101,2324166101013,NARCOTIC ANALGESICS,38,0.0,NaN,2,633.23,True
6465,2322752102,2322752102001,NARCOTIC ANALGESICS,32,0.0,NaN,1,525.59,True
55398,2570267101,2570267101003,ALBUTEROL,30,208.0,16.0,3,1084.40,True
53560,2469255101,2469255101010,NARCOTIC ANALGESICS,30,0.0,NaN,1,268.55,True
9745,2323939102,2323939102028,FREESTYLE,26,631.0,30.0,2,1541.26,True
77835,2579512101,2579512101001,NARCOTIC ANALGESIC COMBINATIONS,24,636.0,30.0,3,6517.91,True
18383,2327119101,2327119101009,ALBUTEROL,24,408.0,17.0,2,1233.60,True


In [80]:
# Example merge: attach person-level predictors from Consolidated
person_covariate_cols = [
    "DUPERSID",
    "PMEDUP42",
    "PMEDUP53",
    "ADMOOD42",
    "ADPAIN42",
    "ADSLEEP42",
    f"INSCOV{yr_suffix}",
    f"MCARE{yr_suffix}",
    f"MCAID{yr_suffix}",
    f"PRIV{yr_suffix}",
    f"AGE{yr_suffix}X",
    "SEX",
    f"POVCAT{yr_suffix}",
    f"RXTOT{yr_suffix}",
]

available_person_cols = [c for c in person_covariate_cols if c in (all_columns.get(cons_file) or read_meps_columns(MEPS_DIR / cons_file)[0])]

person_level = pd.read_excel(
    MEPS_DIR / cons_file,
    sheet_name=0,
    usecols=available_person_cols,
    engine=EXCEL_ENGINE,
)

# One row per person-drug with person traits attached (for regression / grouping)
analysis_df = person_drug.merge(person_level, on="DUPERSID", how="left", validate="m:1")

print(f"analysis_df shape: {analysis_df.shape}")
print("Example: do people reporting problems paying for meds (PMEDUP42) have fewer fills on average?")

if "PMEDUP42" in analysis_df.columns:
    display(
        analysis_df.groupby("PMEDUP42", dropna=False)["n_fills"]
        .agg(["count", "mean", "median"])
        .rename(columns={"count": "person_drug_pairs"})
    )
else:
    print("PMEDUP42 not in consolidated file for this year")

analysis_df.head()

analysis_df shape: (78609, 22)
Example: do people reporting problems paying for meds (PMEDUP42) have fewer fills on average?


,person_drug_pairs,mean,median
PMEDUP42,,,
-8,731,3.518468,2.0
-7,23,5.956522,5.0
-1,5144,1.699844,1.0
1,67461,3.725397,3.0
2,5250,3.234857,2.0


,DUPERSID,DRUGIDX,RXNAME,n_fills,total_days_supplied,median_days_supplied,n_rounds_with_fill,total_oop,multiple_fills,AGE20X,...,ADSLEEP42,ADMOOD42,POVCAT20,INSCOV20,MCAID20,MCARE20,PRIV20,PMEDUP42,PMEDUP53,RXTOT20
0,2320005101,2320005101001,METFORMIN,3,270.0,90.0,1,26.85,True,73,...,-1,-1,2,2,2,1,2,1,-1,3
1,2320005102,2320005102001,LOSARTAN POT,3,270.0,90.0,1,339.03,True,84,...,-1,-1,2,2,2,1,2,1,-1,9
2,2320005102,2320005102002,HYDROCHLOROT,3,270.0,90.0,1,67.37,True,84,...,-1,-1,2,2,2,1,2,1,-1,9
3,2320005102,2320005102003,ATORVASTATIN,3,270.0,90.0,1,157.94,True,84,...,-1,-1,2,2,2,1,2,1,-1,9
4,2320012102,2320012102001,GLIPIZIDE ER,3,0.0,NaN,2,29.00,True,80,...,1,2,3,2,2,1,2,1,1,18


### Decision summary

**Use for non-adherence**
- `h220a` / `h229a` / `h239a` / `h248a` — **outcomes**: `n_fills` per `DRUGIDX`, `RXDAYSUP`, `RXQUANTY`, `PURCHRD`
- Matching Consolidated file — **confounders**: insurance, demographics, `PMEDUP*` (cost barrier)
- Matching Conditions file — **optional**: comorbidity via `LINKIDX` + `CONDIDX`

**Do not expect from MEPS alone**
- Drug-specific **side effects** → use person-level symptom proxies only with caution, or add external adverse-event data
- Gold-standard **PDC/adherence** → no exact fill dates in the public Rx file; gaps in therapy are hard to measure precisely

**Recommended starting pipeline (one year)**
1. Aggregate Rx file to person-drug refill metrics (`analysis_df` above)
2. Merge Consolidated on `DUPERSID`
3. Model `n_fills` or `total_days_supplied` vs `PMEDUP*`, insurance, age, etc.
4. Treat any mood/pain variables as **exploratory person-level proxies**, not causal proof of side effects

## Which column tells you their condition?

**Not in the prescription file (`h220a`).** Condition/diagnosis lives in the **Medical Conditions** file (`h222` for 2020) or as simple yes/no flags in **Consolidated** (`H224`).

| What you need | File | Column | Example |
|---|---|---|---|
| **Specific diagnosis (best)** | `h222.xlsx` | **`ICD10CDX`** | `I10` = hypertension, `E11` = type 2 diabetes |
| Easier grouped label | `h222.xlsx` | **`CCSR1X`** | `CIR007`, `END002` (AHRQ clinical category) |
| Unique condition row | `h222.xlsx` | **`CONDIDX`** | Primary key for that condition record |
| Person with many conditions | `h222.xlsx` | **`DUPERSID`** | Join key (one person → many condition rows) |
| Ever diagnosed? (coarse) | `H224.xlsx` | **`DIABDX_M18`, `CHDDX`, `CANCERDX`, `ASTHDX`** | 1 = yes, 2 = no (person-level, not each diagnosis) |

To connect a **drug** to a **condition**, use `LINKIDX` on the Rx file and MEPS condition linkage (`RXNUM` in 2020; `RXCOND` flag in 2021+ on the conditions file).

In [81]:
# Condition columns — Medical Conditions file (2020 example)
CONDITION_FILE = YEAR_FILE_SETS.loc[YEAR_FILE_SETS["year"] == "2020", "conditions"].iloc[0]

condition_cols = [
    "DUPERSID", "CONDIDX", "CONDN", "CONDRN", "ICD10CDX",
    "CCSR1X", "CCSR2X", "CCSR3X", "AGEDIAG", "INJURY", "RXNUM",
]

conditions = pd.read_excel(
    MEPS_DIR / CONDITION_FILE,
    sheet_name=0,
    usecols=condition_cols,
    engine=EXCEL_ENGINE,
)

# MEPS missing codes are often negative; keep valid ICD-10 rows for display
valid_dx = conditions.loc[conditions["ICD10CDX"].astype(str).str.match(r"^[A-Z][0-9]", na=False)]

print(f"{CONDITION_FILE}: {len(conditions):,} condition rows | {conditions['DUPERSID'].nunique():,} persons")
print("\nSample diagnoses (ICD10CDX = the condition code):")
display(
    valid_dx[["DUPERSID", "CONDIDX", "ICD10CDX", "CCSR1X", "AGEDIAG", "RXNUM"]]
    .head(10)
)

# Person-level chronic condition flags in Consolidated (yes/no, not full diagnosis list)
consolidated_dx_flags = [c for c in all_columns["H224.xlsx"] if c.endswith("DX") or "DIAB" in c.upper()]
consolidated_dx_flags = [c for c in consolidated_dx_flags if c in {"CHDDX", "CANCERDX", "DIABDX_M18", "ASTHDX", "ARTHDX", "HIBPDX"}]

person_flags = pd.read_excel(
    MEPS_DIR / "H224.xlsx",
    usecols=["DUPERSID"] + consolidated_dx_flags,
    engine=EXCEL_ENGINE,
).head(8)

print("\nConsolidated person-level condition flags (1=yes, 2=no):")
display(person_flags)

h222.xlsx: 80,802 condition rows | 18,569 persons

Sample diagnoses (ICD10CDX = the condition code):


,DUPERSID,CONDIDX,ICD10CDX,CCSR1X,AGEDIAG,RXNUM
1,2320005102,2320005102001,I10,CIR007,71,2
2,2320005102,2320005102004,Z13,FAC003,-1,2
3,2320006102,2320006102008,Z04,FAC003,-1,0
4,2320006103,2320006103002,R55,SYM001,-1,0
5,2320012102,2320012102001,I10,CIR007,65,2
6,2320012102,2320012102002,E78,END010,65,2
7,2320012102,2320012102003,E11,END002,65,7
8,2320012102,2320012102004,E07,END001,-1,1
9,2320012102,2320012102008,H26,EYE002,-1,0
10,2320013101,2320013101001,E78,END010,77,3



Consolidated person-level condition flags (1=yes, 2=no):


,DUPERSID,HIBPDX,CHDDX,CANCERDX,DIABDX_M18,ARTHDX,ASTHDX
0,2320005101,2,2,2,2,2,2
1,2320005102,1,2,2,2,1,2
2,2320006101,2,2,2,2,2,2
3,2320006102,2,2,2,2,2,1
4,2320006103,2,2,2,2,2,2
5,2320012102,1,2,2,1,1,2
6,2320013101,2,2,2,2,1,2
7,2320018101,1,2,1,2,1,2


## Part 4 — How the files link to each other

MEPS is a set of **related tables** with different row grains. You join them on shared ID columns — not by stacking columns from every file into one sheet.

```mermaid
flowchart LR
  subgraph person["Person level (1 row per person in Consolidated)"]
    CONS["Full Year Consolidated\nH224 / h233 / h243 / h251"]
  end

  subgraph events["Event level (many rows per person)"]
    RX["Prescribed Medicines\nh220a / h229a / h239a / h248a"]
    COND["Medical Conditions\nh222 / h231 / h241 / h249"]
    PLAN["Person Round Plan\nh223 / h232 / h242 / h250"]
  end

  RX -->|"DUPERSID"| CONS
  COND -->|"DUPERSID"| CONS
  PLAN -->|"DUPERSID"| CONS
  RX -.->|"LINKIDX → CLNK/RXLK link files → CONDIDX"| COND
```

### Link keys at a glance

| Key | Where it appears | Grain | Links what |
|---|---|---|---|
| **`DUPERSID`** | All 4 file types | Person | Primary join key across files |
| **`DUID` + `PID`** | Rx, Conditions, Consolidated | Person | Components of `DUPERSID` |
| **`RXRECIDX`** | Prescribed Meds only | One fill/refill | Primary key within Rx file |
| **`DRUGIDX`** | Prescribed Meds only | Person + drug | Groups multiple fills of same drug |
| **`LINKIDX`** | Prescribed Meds | Person + drug + round | Bridge to conditions **via MEPS link files** |
| **`CONDIDX`** | Medical Conditions | One condition | Primary key within Conditions file |
| **`ICD10CDX`** | Medical Conditions | Diagnosis code | The actual condition label |
| **`PANEL` / `PURCHRD`** | Several files | Survey round | Align round-level plan data to fills |

**Important:** Your folder has the 4 main event/person files but **not** the official **CLNK** (condition–event) and **RXLK** (Rx–event) crosswalk files. So `LINKIDX` does **not** join directly to `CONDIDX` in a single merge — you need those link files for drug-specific condition attribution, or use person-level joins instead.

In [82]:
# Reference table: every cross-file relationship relevant to non-adherence work
file_link_guide = pd.DataFrame(
    [
        {
            "from_file": "Prescribed Medicines",
            "to_file": "Full Year Consolidated",
            "join_keys": "DUPERSID",
            "relationship": "many Rx rows → 1 person row",
            "use_for_adherence": "Attach PMEDUP*, insurance, age/sex, person-level chronic flags (DIABDX, CHDDX)",
        },
        {
            "from_file": "Prescribed Medicines",
            "to_file": "Medical Conditions",
            "join_keys": "DUPERSID (person-level only)",
            "relationship": "many Rx × many condition rows — DO NOT row-merge directly",
            "use_for_adherence": "Aggregate conditions to person first (n_conditions, comorbidity); avoid inflated Cartesian joins",
        },
        {
            "from_file": "Prescribed Medicines",
            "to_file": "Medical Conditions",
            "join_keys": "LINKIDX → CLNK/RXLK → CONDIDX",
            "relationship": "many-to-many (needs link files not in this folder)",
            "use_for_adherence": "Correct way to tie a specific fill to treated condition(s); download year-matched CLNK/RXLK from MEPS",
        },
        {
            "from_file": "Prescribed Medicines",
            "to_file": "Person Round Plan",
            "join_keys": "DUPERSID (+ round: PURCHRD ↔ PANEL when possible)",
            "relationship": "many Rx rows → several plan rows per person",
            "use_for_adherence": "Premiums/deductibles as access barriers; optional if Consolidated already has insurance vars",
        },
        {
            "from_file": "Medical Conditions",
            "to_file": "Full Year Consolidated",
            "join_keys": "DUPERSID",
            "relationship": "many condition rows → 1 person row",
            "use_for_adherence": "Combine detailed ICD-10 list with person demographics from Consolidated",
        },
        {
            "from_file": "Prescribed Medicines (internal)",
            "to_file": "Prescribed Medicines (internal)",
            "join_keys": "DRUGIDX groups RXRECIDX fills",
            "relationship": "1 drug → many fills",
            "use_for_adherence": "Build n_fills, total_days_supplied — your adherence outcomes",
        },
    ]
)

file_link_guide

,from_file,to_file,join_keys,relationship,use_for_adherence
0,Prescribed Medicines,Full Year Consolidated,DUPERSID,many Rx rows → 1 person row,"Attach PMEDUP*, insurance, age/sex, person-lev..."
1,Prescribed Medicines,Medical Conditions,DUPERSID (person-level only),many Rx × many condition rows — DO NOT row-mer...,Aggregate conditions to person first (n_condit...
2,Prescribed Medicines,Medical Conditions,LINKIDX → CLNK/RXLK → CONDIDX,many-to-many (needs link files not in this fol...,Correct way to tie a specific fill to treated ...
3,Prescribed Medicines,Person Round Plan,DUPERSID (+ round: PURCHRD ↔ PANEL when possible),many Rx rows → several plan rows per person,Premiums/deductibles as access barriers; optio...
4,Medical Conditions,Full Year Consolidated,DUPERSID,many condition rows → 1 person row,Combine detailed ICD-10 list with person demog...
5,Prescribed Medicines (internal),Prescribed Medicines (internal),DRUGIDX groups RXRECIDX fills,1 drug → many fills,"Build n_fills, total_days_supplied — your adhe..."


In [83]:
# Demonstrate join cardinalities on 2020 files (run after Part 3 cells)
LINK_YEAR = "2020"
rx_path = MEPS_DIR / YEAR_FILE_SETS.loc[YEAR_FILE_SETS["year"] == LINK_YEAR, "prescribed_meds"].iloc[0]
cond_path = MEPS_DIR / YEAR_FILE_SETS.loc[YEAR_FILE_SETS["year"] == LINK_YEAR, "conditions"].iloc[0]
cons_path = MEPS_DIR / YEAR_FILE_SETS.loc[YEAR_FILE_SETS["year"] == LINK_YEAR, "consolidated"].iloc[0]
plan_path = MEPS_DIR / YEAR_FILE_SETS.loc[YEAR_FILE_SETS["year"] == LINK_YEAR, "person_plan"].iloc[0]

rx_link = pd.read_excel(
    rx_path,
    usecols=["DUPERSID", "RXRECIDX", "DRUGIDX", "LINKIDX", "PURCHRD", "RXNAME"],
    engine=EXCEL_ENGINE,
)
cond_link = pd.read_excel(
    cond_path,
    usecols=["DUPERSID", "CONDIDX", "ICD10CDX", "RXNUM"],
    engine=EXCEL_ENGINE,
)
cons_link = pd.read_excel(cons_path, usecols=["DUPERSID", "PMEDUP42", "DIABDX_M18"], engine=EXCEL_ENGINE)
plan_link = pd.read_excel(plan_path, usecols=["DUPERSID", "PANEL", "PREMLEVX"], engine=EXCEL_ENGINE)

# 1) SAFE: person-drug outcomes + consolidated person traits (many-to-one)
person_drug_link = (
    rx_link.groupby(["DUPERSID", "DRUGIDX", "RXNAME"], as_index=False)
    .agg(n_fills=("RXRECIDX", "nunique"), n_link_groups=("LINKIDX", "nunique"))
)
merged_cons = person_drug_link.merge(cons_link, on="DUPERSID", how="left", validate="m:1")

# 2) SAFE: aggregate conditions to person before joining
cond_by_person = cond_link.groupby("DUPERSID", as_index=False).agg(
    n_conditions=("CONDIDX", "nunique"),
    n_medication_linked_conditions=("RXNUM", lambda s: (s > 0).sum()),
)
merged_cond = person_drug_link.merge(cond_by_person, on="DUPERSID", how="left", validate="m:1")

# 3) UNSAFE: row-level Rx × Conditions on DUPERSID alone (shows why not to do this)
naive_bad = rx_link.merge(cond_link, on="DUPERSID", how="inner")
inflation = len(naive_bad) / len(rx_link)

# 4) Direct LINKIDX = CONDIDX does not work without CLNK/RXLK files
direct_link_try = rx_link.merge(cond_link, left_on="LINKIDX", right_on="CONDIDX", how="inner")

join_diagnostics = pd.DataFrame(
    [
        {
            "join": "person_drug → Consolidated on DUPERSID",
            "result_rows": len(merged_cons),
            "expected": "Same as person_drug rows (one person trait set repeated per drug)",
            "status": "OK",
        },
        {
            "join": "person_drug → Conditions (aggregated) on DUPERSID",
            "result_rows": len(merged_cond),
            "expected": "Same as person_drug rows",
            "status": "OK",
        },
        {
            "join": "Rx rows × Condition rows on DUPERSID only",
            "result_rows": len(naive_bad),
            "expected": f"Inflated ~{inflation:.1f}× vs Rx rows — avoid",
            "status": "BAD",
        },
        {
            "join": "Rx LINKIDX = Conditions CONDIDX (no link files)",
            "result_rows": len(direct_link_try),
            "expected": "0 matches — need CLNK/RXLK crosswalk",
            "status": "Needs link files",
        },
    ]
)

print("Unique persons in each 2020 file:")
print(
    pd.Series(
        {
            "Rx fills (rows)": len(rx_link),
            "Rx persons": rx_link["DUPERSID"].nunique(),
            "Condition rows": len(cond_link),
            "Condition persons": cond_link["DUPERSID"].nunique(),
            "Consolidated persons": cons_link["DUPERSID"].nunique(),
            "Plan rows": len(plan_link),
            "Plan persons": plan_link["DUPERSID"].nunique(),
        }
    )
)
join_diagnostics

Unique persons in each 2020 file:
Rx fills (rows)         279755
Rx persons               15743
Condition rows           80802
Condition persons        18569
Consolidated persons     27805
Plan rows                45214
Plan persons             16248
dtype: int64


,join,result_rows,expected,status
0,person_drug → Consolidated on DUPERSID,78609,Same as person_drug rows (one person trait set...,OK
1,person_drug → Conditions (aggregated) on DUPERSID,78609,Same as person_drug rows,OK
2,Rx rows × Condition rows on DUPERSID only,2323124,Inflated ~8.3× vs Rx rows — avoid,BAD
3,Rx LINKIDX = Conditions CONDIDX (no link files),0,0 matches — need CLNK/RXLK crosswalk,Needs link files


In [84]:
# Recommended linked dataset for non-adherence analysis (person-drug grain)
# Uses only safe joins available with the files in this folder

enriched_adherence = (
    person_drug_link
    .merge(cons_link, on="DUPERSID", how="left", validate="m:1")
    .merge(cond_by_person, on="DUPERSID", how="left", validate="m:1")
)

# Optional: attach round-level premium (collapse plan to person mean — plan has ~3 rows/person)
plan_by_person = plan_link.groupby("DUPERSID", as_index=False).agg(avg_premium=("PREMLEVX", "mean"))
enriched_adherence = enriched_adherence.merge(plan_by_person, on="DUPERSID", how="left", validate="m:1")

print(f"Grain: one row per person-drug | shape = {enriched_adherence.shape}")
print("\nHow each link helps non-adherence analysis:")
help_summary = pd.DataFrame(
    [
        ("n_fills / DRUGIDX", "Outcome: refill frequency & gaps proxy"),
        ("PMEDUP42 (Consolidated)", "Predictor: cost barrier → non-adherence"),
        ("DIABDX_M18 (Consolidated)", "Stratify: chronic disease group adherence patterns"),
        ("n_conditions (Conditions agg)", "Confounder: comorbidity burden"),
        ("n_medication_linked_conditions", "Severity proxy: more treated conditions"),
        ("avg_premium (Plan agg)", "Confounder: insurance cost exposure"),
    ],
    columns=["field", "role_in_analysis"],
)
display(help_summary)

display(
    enriched_adherence.sort_values("n_fills", ascending=False)
    .head(8)[
        [
            "DUPERSID", "RXNAME", "n_fills", "PMEDUP42", "DIABDX_M18",
            "n_conditions", "n_medication_linked_conditions", "avg_premium",
        ]
    ]
)

Grain: one row per person-drug | shape = (78609, 10)

How each link helps non-adherence analysis:


,field,role_in_analysis
0,n_fills / DRUGIDX,Outcome: refill frequency & gaps proxy
1,PMEDUP42 (Consolidated),Predictor: cost barrier → non-adherence
2,DIABDX_M18 (Consolidated),Stratify: chronic disease group adherence patt...
3,n_conditions (Conditions agg),Confounder: comorbidity burden
4,n_medication_linked_conditions,Severity proxy: more treated conditions
5,avg_premium (Plan agg),Confounder: insurance cost exposure


,DUPERSID,RXNAME,n_fills,PMEDUP42,DIABDX_M18,n_conditions,n_medication_linked_conditions,avg_premium
45341,2466612102,ONETOUCH,106,1,1,7.0,6.0,2.333333
73276,2577708104,NARCOTIC ANALGESICS,60,1,2,2.0,2.0,NaN
38707,2464502101,NARCOTIC ANALGESICS,40,1,2,5.0,4.0,1.000000
10315,2324166101,NARCOTIC ANALGESICS,38,1,2,11.0,9.0,NaN
6465,2322752102,NARCOTIC ANALGESICS,32,1,2,8.0,5.0,NaN
55398,2570267101,ALBUTEROL,30,1,2,1.0,1.0,0.666667
53560,2469255101,NARCOTIC ANALGESICS,30,2,2,5.0,4.0,1.666667
9745,2323939102,FREESTYLE,26,1,1,15.0,8.0,NaN


### How linking supports non-adherence analysis

| Step | Join | What you learn |
|---|---|---|
| **1. Build outcomes** | Inside Rx file via `DRUGIDX` / `RXRECIDX` | Fill counts, days supplied — your adherence measures |
| **2. Add access barriers** | Rx → Consolidated on `DUPERSID` | Does `PMEDUP*` (problems paying) predict fewer refills? |
| **3. Add demographics & insurance** | Rx → Consolidated on `DUPERSID` | Control for age, coverage, poverty (`POVCAT*`, `INSCOV*`) |
| **4. Add comorbidity** | Rx → Conditions **aggregated** on `DUPERSID` | More conditions → more meds/complexity → different adherence |
| **5. Drug ↔ condition (advanced)** | `LINKIDX` + CLNK/RXLK → `CONDIDX` | Which diagnosis explains a specific drug (e.g. metformin ↔ diabetes) — **requires downloading MEPS link files** |
| **6. Round-level insurance (optional)** | Rx `PURCHRD` + Plan `PANEL` on `DUPERSID` | Premium/deductible changes within the year |

**Pipeline to run:** Part 3 setup → Part 4 join diagnostics → `enriched_adherence` table → model `n_fills` (or `total_days_supplied`) against `PMEDUP*`, comorbidity counts, and drug class (`TC1` from Rx file).

## Part 4b — How **each Excel file** links to **each other** (file-by-file)

MEPS gives you **4 workbooks per calendar year**. Only link files from the **same year** (e.g. all 2020 files together). Do not join `h220a.xlsx` (2020) to `h229a.xlsx` (2021) in one analysis row unless you are doing a deliberate multi-year panel study.

Below: every **pair** of files for **2020**, then a table for all 16 files in this folder.

### 2020 — `h220a.xlsx` (Prescribed Medicines)

| Links to | Join column(s) | Relationship | What you get |
|---|---|---|---|
| **`H224.xlsx`** (Consolidated) | `h220a.DUPERSID` = `H224.DUPERSID` | Many Rx rows → **1** person row | Age, sex, insurance, `PMEDUP*`, `DIABDX_M18`, `CHDDX`, weights |
| **`h222.xlsx`** (Conditions) | `DUPERSID` only (safe if you **aggregate** conditions first) | Many Rx × many conditions per person | Comorbidity counts — **do not** row-merge without aggregating |
| **`h222.xlsx`** (Conditions) | `h220a.LINKIDX` → **CLNK/RXLK files** → `h222.CONDIDX` | Many-to-many (needs extra MEPS files **not in this folder**) | Which ICD-10 condition goes with which drug fill |
| **`h223.xlsx`** (Person Plan) | `h220a.DUPERSID` = `h223.DUPERSID` (+ round: `PURCHRD` ↔ `PANEL` when aligned) | Many Rx rows → **~3** plan rows per person | Premiums, deductibles, plan type |

**Internal (within `h220a` only):** `DRUGIDX` groups rows; `RXRECIDX` is unique per fill.

---

### 2020 — `H224.xlsx` (Full Year Consolidated)

| Links to | Join column(s) | Relationship | What you get |
|---|---|---|---|
| **`h220a.xlsx`** | `H224.DUPERSID` = `h220a.DUPERSID` | **1** person row → many Rx rows | Attach person traits to every fill or to person-drug summaries |
| **`h222.xlsx`** | `H224.DUPERSID` = `h222.DUPERSID` | **1** person row → many condition rows | Person demographics + full condition list (aggregate `h222` first for modeling) |
| **`h223.xlsx`** | `H224.DUPERSID` = `h223.DUPERSID` | **1** person row → several plan rows | Extra insurance detail (often redundant with `H224`) |

**Does not contain:** `RXRECIDX`, `LINKIDX`, `CONDIDX`, or `ICD10CDX`.

---

### 2020 — `h222.xlsx` (Medical Conditions)

| Links to | Join column(s) | Relationship | What you get |
|---|---|---|---|
| **`h220a.xlsx`** | `h222.DUPERSID` = `h220a.DUPERSID` (person-level) | Many conditions → many Rx rows per person | Person-level comorbidity + refill outcomes |
| **`h220a.xlsx`** | `CONDIDX` via **CLNK/RXLK** + `h220a.LINKIDX` | Many-to-many (needs link files) | Drug-specific indication |
| **`H224.xlsx`** | `h222.DUPERSID` = `H224.DUPERSID` | Many condition rows → **1** person row | `ICD10CDX` + `DIABDX_M18` / `CHDDX` flags on same person |
| **`h223.xlsx`** | `h222.DUPERSID` = `h223.DUPERSID` | Many conditions → several plan rows | Rarely needed for adherence |

**Condition text column:** `ICD10CDX` (not in Rx or Consolidated as a list).

---

### 2020 — `h223.xlsx` (Person Round Plan)

| Links to | Join column(s) | Relationship | What you get |
|---|---|---|---|
| **`h220a.xlsx`** | `h223.DUPERSID` = `h220a.DUPERSID` | Several plan rows → many Rx rows | Round-level premiums vs fills in that round |
| **`H224.xlsx`** | `h223.DUPERSID` = `H224.DUPERSID` | Several plan rows → **1** person row | Most insurance variables already on `H224` |
| **`h222.xlsx`** | `h223.DUPERSID` = `h222.DUPERSID` | Several plan rows → many condition rows | Uncommon for adherence work |

**Does not contain:** prescription fills (`RXRECIDX`) or diagnosis codes (`ICD10CDX`).

In [ ]:
# Pairwise link matrix for every Excel file in this folder (same calendar year only)

def links_for_year(year: str) -> pd.DataFrame:
    row = YEAR_FILE_SETS.loc[YEAR_FILE_SETS["year"] == year].iloc[0]
    rx, cons, cond, plan = row["prescribed_meds"], row["consolidated"], row["conditions"], row["person_plan"]
    files = [rx, cons, cond, plan]
    labels = {
        rx: "Rx (fills)",
        cons: "Consolidated (person)",
        cond: "Conditions (dx)",
        plan: "Plan (insurance round)",
    }

    # Directed links: (from_file, to_file, keys, note)
    edges = [
        (rx, cons, "DUPERSID", "many fills → 1 person"),
        (cons, rx, "DUPERSID", "1 person → many fills"),
        (rx, cond, "DUPERSID", "many fills → many conditions (aggregate cond first)"),
        (cond, rx, "DUPERSID", "many conditions → many fills (aggregate cond first)"),
        (rx, cond, "LINKIDX → CLNK/RXLK → CONDIDX", "precise drug–condition (link files NOT in folder)"),
        (rx, plan, "DUPERSID (+ PURCHRD↔PANEL)", "many fills → ~3 plan rows/person"),
        (plan, rx, "DUPERSID", "plan rows → many fills"),
        (cons, cond, "DUPERSID", "1 person → many conditions"),
        (cond, cons, "DUPERSID", "many conditions → 1 person"),
        (cons, plan, "DUPERSID", "1 person → several plan rows"),
        (plan, cons, "DUPERSID", "several plan rows → 1 person"),
        (cond, plan, "DUPERSID", "many conditions → several plan rows"),
        (plan, cond, "DUPERSID", "several plan rows → many conditions"),
    ]

    records = []
    for f_from, f_to, keys, note in edges:
        records.append(
            {
                "year": year,
                "from_file": f_from,
                "from_type": labels[f_from],
                "to_file": f_to,
                "to_type": labels[f_to],
                "join_keys": keys,
                "relationship": note,
            }
        )
    return pd.DataFrame(records)


all_file_links = pd.concat([links_for_year(y) for y in YEAR_FILE_SETS["year"]], ignore_index=True)

print("Every directed link between the 4 files in each year (12 links × 4 years = 48 rows):\n")
display(
    all_file_links.sort_values(["year", "from_file", "to_file"]).reset_index(drop=True)
)

In [ ]:
# Compact matrix: can file A join to file B in the same year? (via DUPERSID)

years = YEAR_FILE_SETS["year"].tolist()
file_cols = ["prescribed_meds", "consolidated", "conditions", "person_plan"]
short = {"prescribed_meds": "Rx", "consolidated": "Cons", "conditions": "Cond", "person_plan": "Plan"}

for year in years:
    row = YEAR_FILE_SETS.loc[YEAR_FILE_SETS["year"] == year].iloc[0]
    names = [row[c] for c in file_cols]
    matrix = pd.DataFrame(index=[short[c] for c in file_cols], columns=[short[c] for c in file_cols])

    for i, fc_i in enumerate(file_cols):
        for j, fc_j in enumerate(file_cols):
            if i == j:
                matrix.iloc[i, j] = "—"
            else:
                fi, fj = row[fc_i], row[fc_j]
                if fc_i == "prescribed_meds" and fc_j == "conditions":
                    matrix.iloc[i, j] = "DUPERSID*\n(+CLNK†)"
                elif fc_i == "conditions" and fc_j == "prescribed_meds":
                    matrix.iloc[i, j] = "DUPERSID*\n(+CLNK†)"
                else:
                    matrix.iloc[i, j] = "DUPERSID"

    print(f"\n=== {year} — join key between files (rows = FROM, cols = TO) ===")
    print("Files:", dict(zip([short[c] for c in file_cols], names)))
    display(matrix)

print("\n* DUPERSID only: safe if you aggregate to person or person-drug before merging wide tables.")
print("† CLNK/RXLK: MEPS link crosswalk files (download separately) for LINKIDX ↔ CONDIDX.")

### Cross-year: how the 16 files relate to each other

| From | To | Link? | Rule |
|---|---|---|---|
| `h220a` (2020 Rx) | `h229a` (2021 Rx) | `DUPERSID` | Same **file type**, different years — only for longitudinal/panel studies; column names change (`PERWT20F` → `PERWT21F`) |
| `h220a` (2020 Rx) | `H224` (2020 Cons) | `DUPERSID` | **Yes** — same year |
| `h220a` (2020 Rx) | `h233` (2021 Cons) | — | **No** — different survey years; do not join |
| Any 2020 file | Any 2023 file | — | **No** — treat each year as a separate dataset |

**Files in this folder that never link directly to each other**

| File A | File B | Why there is no single-column join |
|---|---|---|
| `h220a` | `h222` | No shared row ID; `LINKIDX` ≠ `CONDIDX`. Use `DUPERSID` (person) or CLNK/RXLK (drug–condition). |
| `h223` | `h222` | Only `DUPERSID` (+ `PANEL`); no prescription or diagnosis columns on plan file. |

### For non-adherence: minimum file set per year

```
h220a  ──DUPERSID──►  H224        (required)
h220a  ──DUPERSID──►  h222        (optional; aggregate conditions first)
h220a  ──DUPERSID──►  h223        (optional; insurance detail)
```